## 1. What is an RDD?

**RDD (Resilient Distributed Dataset)** is Spark's original distributed data structure.

It is an **immutable**, **distributed**, and **fault-tolerant** collection of objects processed in parallel across a cluster.

| Feature | Description |
|---|---|
| **Distributed** | Spread across multiple executors |
| **Immutable** | Cannot be modified after creation |
| **Fault Tolerant** | Recovers automatically using Lineage |
| **Lazy** | Executes only when an Action is called |

## 2. Why was RDD Introduced?

Before Spark, **Hadoop MapReduce** wrote intermediate results to disk after every stage.

**Problems:**
- High Disk I/O
- Slow Processing
- Increased Execution Time

**RDD solved this by:**
- Processing data **in memory**
- Executing operations **in parallel**
- Recovering automatically from failures

## 4. RDD Characteristics

### 4.1 Immutable
Every transformation creates a **new RDD** — the original is never changed.

### 4.2 Distributed
RDD is split into partitions, each processed by a different executor.

### 4.3 Lazy Evaluation
Transformations are recorded but **not executed** until an Action is called.

### 4.4 Fault Tolerance
If a partition is lost, Spark **replays the lineage** to recreate it — no duplicate copies needed.

## 5. What is Lineage?

**Lineage** is the history of all transformations used to build an RDD.
If a partition is lost, Spark replays these steps to recreate it automatically.

> This is how Spark achieves fault tolerance **without** storing extra copies of data.

## 6. Creating an RDD

In [0]:
# Method 1: From a Python collection
numbers = [1, 2, 3, 4, 5]
rdd = sc.parallelize(numbers)
print("✅ RDD created from Python list")
print("Partitions:", rdd.getNumPartitions())

In [0]:
# Method 2: From a text file
# rdd = sc.textFile("/FileStore/data/sample.txt")
# print(rdd.getNumPartitions())

# Using inline data for demo
textRDD = sc.parallelize(["Spark is fast", "Spark is powerful", "RDD is distributed"])
print("✅ RDD created from text data")

## 7. RDD Transformations

Transformations create a **new RDD** — they do NOT execute immediately.

| Transformation | Description |
|---|---|
| `map()` | Apply a function to each element |
| `filter()` | Filter records by condition |
| `flatMap()` | Split elements into multiple values |
| `distinct()` | Remove duplicates |
| `union()` | Combine two RDDs |
| `sortBy()` | Sort data |
| `groupByKey()` | Group values by key |
| `reduceByKey()` | Aggregate values by key |

In [0]:
# 🔧 TRANSFORMATIONS — Nothing executes here

numbers = sc.parallelize([1, 2, 3, 4, 5])

squareRDD = numbers.map(lambda x: x * x)       # No execution
evenRDD = squareRDD.filter(lambda x: x > 4)    # No execution

print("✅ Transformations recorded. Execution has NOT started yet.")

## 8. RDD Actions

Actions **trigger execution** of the full plan.

| Action | Description |
|---|---|
| `collect()` | Return all records as a list |
| `count()` | Count total records |
| `first()` | Return first record |
| `take(n)` | Return first N records |
| `reduce()` | Aggregate all values |
| `saveAsTextFile()` | Save output to storage |

In [0]:
# ⚡ ACTION — Triggers full execution now!

print("🚀 Action called — Spark executes now!")
result = evenRDD.collect()
print("Result:", result)

## 9. Narrow vs Wide Transformations

### Narrow Transformation
Each child partition depends on **only one** parent partition — no shuffle.

Examples: `map()`, `filter()`, `union()`

### Wide Transformation
Child partitions depend on **multiple** parent partitions — causes shuffle.

Examples: `groupByKey()`, `reduceByKey()`, `join()`, `sortByKey()`
> Minimize wide transformations to reduce shuffle and improve performance.

In [0]:
# STEP 1 — Create RDD
numbers = sc.parallelize([1, 2, 3, 4, 5, 6, 7, 8, 9, 10])
print("Step 1 ✅ RDD created — no execution yet")

# STEP 2 — Transformations
evenRDD = numbers.filter(lambda x: x % 2 == 0)
squareRDD = evenRDD.map(lambda x: x * x)
print("Step 2 ✅ Transformations recorded — no execution yet")

# STEP 3 — Action triggers execution
print("Step 3 ⚡ collect() called — Spark executes everything NOW!")
result = squareRDD.collect()
print("Result:", result)

## 11. Word Count Example

Classic Spark example — count occurrences of each word across multiple lines.

In [0]:
# Word Count using RDD

textRDD = sc.parallelize([
    "Spark is fast",
    "Spark is powerful",
    "RDD is distributed"
])

wordCount = (
    textRDD
    .flatMap(lambda line: line.split())       # Split each line into words
    .map(lambda word: (word, 1))              # Map each word to (word, 1)
    .reduceByKey(lambda a, b: a + b)          # Sum counts per word
)

print("⚡ Action triggered!")
result = wordCount.collect()
for word, count in sorted(result):
    print(f"  {word}: {count}")

## 12. Partitioning & Caching

### Partitioning
Partitions allow Spark to process data **in parallel**.
Each partition is handled by one executor independently.

### Caching
Without caching, Spark **recomputes** the RDD every time an action is called.
Cache when you reuse an RDD multiple times.

In [0]:
# Check partitions
rdd = sc.parallelize(range(1, 101))
print("Default partitions:", rdd.getNumPartitions())

# Custom partition count
rdd4 = sc.parallelize(range(1, 101), 4)
print("Custom partitions:", rdd4.getNumPartitions())

# Cache the RDD — avoids recomputation on repeated actions
rdd4.cache()
print("✅ RDD cached")

print("Count:", rdd4.count())   # Uses cache
print("First:", rdd4.first())   # Uses cache again — no recomputation

## 13. RDD vs DataFrame

| Feature | RDD | DataFrame |
|---|---|---|
| Data Structure | Objects | Rows & Columns |
| Schema | ❌ No | ✅ Yes |
| Catalyst Optimizer | ❌ No | ✅ Yes |
| Performance | Good | Better |
| Ease of Use | Low | High |
| Best For | Low-level processing | ETL & Analytics |

### When to use RDD?
- Low-level distributed programming
- Custom partitioning or algorithms
- Unstructured data processing
- Fine-grained control required

> For most ETL, analytics, and SQL workloads — **use DataFrames**.

%md
![RDD](/Volumes/sample/demo/images/Spark_RDD.png)